In [6]:
# imports
import os
import gzip
import zipfile
import json
import csv
import pandas as pd

In [8]:
print(os.getcwd())
print(os.listdir())

/home/jupyter-sc2641/Varshini_FP
['.ipynb_checkpoints', 'file-manager?origin_id=e524969c-7dff-474c-899c-efddf8d15b83', '3DLNews2.zip', 'exp.ipynb', 'wget-log', '3DLNews2']


In [12]:
# extract csvs from .zip file

# first: unzip .zip folder
with zipfile.ZipFile('/home/jupyter-sc2641/Varshini_FP/3DLNews2.zip', 'r') as zipped_path: 
        zipped_path.extractall('.')
        #print(zipped_path.namelist())

In [45]:
# folder structure:  Google/Twitter -> 1-Newspapers/2-Radio/3-TV/4-Broadcast -> state/preprocessed_state -> jsonl.gz files
# subfolders' info
platform_type = ['1-Google', '2-Twitter']
media_type = ['1-Newspaper', '2-Radio', '3-TV', '4-Broadcast']
fields_to_extract = ['id', 'file_path', 'link', 'publication_date', 'title', 'content', 'location']
years_needed = [2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023]


# function to get to and convert each .jsonl.gz file to .csv
# NOTE: adapted this from extract_data.py script provided by data source
def jsonlgz_to_csv(path, platform_type, media_type, years_needed = None):
    
    print(f"Starting extraction for {path}")

    # input directory
    input_dir = os.path.join(path, 'preprocessed_state')

    if not os.path.exists(input_dir):
        print(f"Warning! Skipping {platform_type} - directory not found: {input_dir}")

    # setting up output directory and individual paths
    output_dir = os.path.join('results', platform_type)
    os.makedirs(output_dir, exist_ok = True)
    output_csv_path = os.path.join(output_dir, f"{media_type}.csv")
    all_extracted_rows = []

    #
    for state in sorted(os.listdir(input_dir)):
        state_path = os.path.join(input_dir, state)
        if not os.path.isdir(state_path):
            continue

        # 
        file_list = os.listdir(state_path)
        print(f"Processing state {state} with {len(file_list)} files")

        for i, file_name in enumerate(file_list):
            if years_needed:
                year_match = next((str(y) for y in years_needed if str(y) in file_name), None)
                if not year_match: 
                    continue
            
            file_path = os.path.join(state_path, file_name)
            
            try: 
                with gzip.open(file_path, 'rt', encoding = 'utf-8') as f: 
                    for line in f:
                        try: 
                            record = json.loads(line)
        
                            if not record.get("is_news_article"):
                                continue
        
                            row = {'id': record.get('id'),
                                   'file_path': file_path}
        
                            for field in fields_to_extract: 
                                if field not in row: 
                                    row[field] = record.get(field, "")
        
                            all_extracted_rows.append(row)
                            
                        except json.JSONDecodeError:
                            continue
        
            except Exception as e:
                print(f"ERROR! Failed to process {file_path}: {e}")

    # 
    if all_extracted_rows:
        print(f"[INFO] Writing total {len(all_extracted_rows)} records to CSV: {output_csv_path}")
        with open(output_csv_path, 'w', newline = '', encoding = 'utf-8') as csv_file:
            writer = csv.DictWriter(csv_file, fieldnames = fields_to_extract)
            writer.writeheader()
            writer.writerows(all_extracted_rows)
    else:
        print(f"[INFO] No valid news articles found for: {platform}/{media_type}")

In [46]:
print("[INFO] Starting extraction process...")
for platform in platform_type:
    print(f"For {platform}:")
    media_root = os.path.join("/home/jupyter-sc2641/Varshini_FP/3DLNews2", platform)
    print(media_root)
    if not os.path.exists(media_root):
        print(f"[WARN] Skipping missing platform directory: {media_root}")
        continue

    for media in media_type:
        print(f"For {media}:")
        full_media_path = os.path.join(media_root, media)
        print(full_media_path)
        print(f"\n[INFO] Scanning platform '{platform}', media folder '{media}'")
        jsonlgz_to_csv(full_media_path, platform, media, years_needed)

print("\n[INFO] Extraction complete.")

[INFO] Starting extraction process...
For 1-Google:
/home/jupyter-sc2641/Varshini_FP/3DLNews2/1-Google
For 1-Newspaper:
/home/jupyter-sc2641/Varshini_FP/3DLNews2/1-Google/1-Newspaper

[INFO] Scanning platform '1-Google', media folder '1-Newspaper'
Starting extraction for /home/jupyter-sc2641/Varshini_FP/3DLNews2/1-Google/1-Newspaper
Processing state AK with 30 files
Processing state AL with 30 files
Processing state AR with 30 files
Processing state AZ with 30 files
Processing state CA with 30 files
Processing state CO with 30 files
Processing state CT with 30 files
Processing state DC with 30 files
Processing state DE with 30 files
Processing state FL with 30 files
Processing state GA with 30 files
Processing state HI with 30 files
Processing state IA with 30 files
Processing state ID with 30 files
Processing state IL with 30 files
Processing state IN with 30 files
Processing state KS with 30 files
Processing state KY with 30 files
Processing state LA with 30 files
Processing state MA